In [1]:
import tonic
import torch
import torch.nn as nn
import random
import pandas as pd
import numpy as np

c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

SEED = 100
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(device)

cuda


# Data Loading

transform_train = tonic.transforms.Compose([
    tonic.transforms.RandomCrop(sensor_size=(120,90,2), target_size=(96,80,2)),
    tonic.transforms.RandomFlipLR(sensor_size=(96,80,2), p = 0.5),
    tonic.transforms.EventDrop(sensor_size=(96,80,2)),
    tonic.transforms.ToFrame(sensor_size=(96,80,2), n_time_bins=30)
])

transform_val = tonic.transforms.Compose([
    tonic.transforms.CenterCrop(sensor_size=(120,90,2), size=(96,80,2)),
    tonic.transforms.ToFrame(sensor_size=(96,80,2), n_time_bins=30)
])

In [3]:
transform_dataset = tonic.transforms.Compose([
    tonic.transforms.Downsample(sensor_size=(240,180,2), target_size=(70,50))
])

transform_train = tonic.transforms.Compose([
    tonic.transforms.RandomCrop(sensor_size=(70,50,2), target_size=(60,45)),
    tonic.transforms.RandomFlipLR(sensor_size=(60,45,2), p = 0.5),
    tonic.transforms.EventDrop(sensor_size=(60,45,2)),
    tonic.transforms.ToFrame(sensor_size=(60,45,2), n_time_bins=30)
])

transform_val = tonic.transforms.Compose([
    tonic.transforms.CenterCrop(sensor_size=(70,50,2), size=(60,45)),
    tonic.transforms.ToFrame(sensor_size=(60,45,2), n_time_bins=30)
])

In [4]:
dataset = tonic.datasets.NCALTECH101(save_to = './Data_60_45_' , transform=transform_dataset)

In [5]:
#changing the targets to numerical values:

targets_series = pd.Series(dataset.targets)
unique_targets = targets_series.unique()
classes = {cls:i for i, cls in enumerate(unique_targets)}
dataset.targets = targets_series.map(classes).to_list()

In [6]:
from torch.utils.data import random_split, WeightedRandomSampler

In [7]:
train = int(len(dataset) * 0.8)
val = int(len(dataset) * 0.1)
test = len(dataset) - train - val

train, val, test = random_split(dataset, [train, val, test])
len(train), len(val), len(test)

(6967, 870, 872)

In [8]:
#initializing balanced weights for samples:

train_series = pd.Series(targets_series[train.indices])
train_counts = train_series.value_counts()
weights = 1 / train_counts
sample_weights = [weights[i] for i in train_series]
len(sample_weights)

6967

In [9]:
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=20000,
    replacement=True
)

In [10]:
batch_size = 64

In [11]:
from torch.utils.data import DataLoader
from tonic.cached_dataset import MemoryCachedDataset

train_cached = tonic.MemoryCachedDataset(train, transform=transform_train)
val_cached = tonic.MemoryCachedDataset(val, transform=transform_val)
test_cached = tonic.MemoryCachedDataset(test, transform=transform_val)

train_loader = DataLoader(train_cached, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                   sampler=sampler, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_cached, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False), pin_memory=True)
test_loader = DataLoader(test_cached, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),pin_memory=True)

In [12]:
for idx, (sample, target) in list(train_cached.samples_dict.items()):
    if isinstance(sample, np.ndarray) and sample.dtype.names is not None:
        raw_sample, target = train[idx]
        train_cached.samples_dict[idx] = (transform_train(raw_sample), target)

In [13]:
damaged = []

for idx, (sample, target) in train_cached.samples_dict.items():
    try:
        torch.tensor(sample)
    except Exception:
        damaged.append(idx)

print("Damaged:", len(damaged))
#can't convert np.ndarray of type numpy.void

Damaged: 0


In [14]:
len(train_loader) * batch_size, len(val_loader) * batch_size, len(test_loader) * batch_size

(19968, 896, 896)

In [15]:
event, target = train_cached[200] 
print(f'Output: {event.shape} for target {target}')

Output: (30, 2, 45, 60) for target 0


# Visualization

import matplotlib.pyplot as plt
import math


def visualize_input_samples(dataset, num_samples=4):
    # Pick random samples
    indices = random.sample(range(len(dataset)), num_samples)

    # Make a grid
    grid_size = math.ceil(math.sqrt(num_samples))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(8, 8))
    axes = np.array(axes).flatten()

    shown = 0

    for idx in indices:
        events, label = dataset[idx]

        if hasattr(events, "detach"):
            events = events.detach().cpu().numpy()

        if events.ndim != 4 or events.sum() == 0:
            continue

        frame = events.sum(axis=0)
        frame = frame[1] - frame[0]

        vmax = max(abs(frame.min()), abs(frame.max()), 1)

        axes[shown].imshow(
            frame,
            cmap="bwr",
            origin="upper",
            vmin=-vmax,
            vmax=vmax,
            interpolation="nearest"
        )

        axes[shown].set_title(f"Sample {idx} | Label: {label}")
        axes[shown].axis("off")

        shown += 1
        if shown == num_samples:
            break

    for i in range(shown, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

visualize_input_samples(val_cached, 4)

# ConvNet

In [16]:
from spikingjelly.activation_based import neuron, functional, surrogate, layer

In [33]:
class ConvNet(nn.Module):
  def __init__(self, in_ch, out_ch, beta, threshold):
    super().__init__()

    tau = 1.0 / (1.0 - beta)

    self.conv1 = layer.Conv2d(in_ch, out_ch, kernel_size = 3, padding = "same", step_mode='m')
    #self.bn1 = layer.GroupNorm(num_groups=4, num_channels=out_ch, step_mode='m')
    self.mp1 = layer.AvgPool2d(2, step_mode='m')
    self.lif1 = neuron.LIFNode(tau=tau, v_threshold=threshold, step_mode='m', 
                               surrogate_function=surrogate.ATan(alpha=0.5), v_reset = None, backend='torch')

    self.conv2 = layer.Conv2d(out_ch, out_ch * 2, kernel_size = 3, padding = "same", step_mode='m')
    #self.bn2 = layer.GroupNorm(num_groups=8, num_channels=out_ch*2, step_mode='m')
    self.mp2 = layer.AvgPool2d(2, step_mode='m')
    self.lif2 = neuron.LIFNode(tau=tau, v_threshold=threshold, step_mode='m', 
                               surrogate_function=surrogate.ATan(alpha=0.5), v_reset = None, backend='torch')

    self.adap = layer.AdaptiveAvgPool2d((2, 2), step_mode='m')
    self.flatten = layer.Flatten(step_mode='m')
    self.fc2 = layer.Linear(out_ch * 2 * 2 * 2, 101, step_mode='m')
    self.lif5 = neuron.LIFNode(tau=tau, v_threshold=threshold, step_mode='m', 
                               surrogate_function=surrogate.ATan(alpha=0.5), v_reset = None, backend='torch')

  def forward(self, x):

    cur1 = self.conv1(x)
    #bn1 = self.bn1(cur1)
    mp1 = self.mp1(cur1)
    spk1 = self.lif1(mp1)

    cur2 = self.conv2(spk1)
    #bn2 = self.bn2(cur2)
    mp2 = self.mp2(cur2)
    spk2 = self.lif2(mp2)
    
    pool = self.adap(spk2)
    flat_spk2 = self.flatten(pool)
    cur5 = self.fc2(flat_spk2)
    spk5= self.lif5(cur5)

    return spk5

# Accuracy functions

In [36]:
def accuracy_temporal_custom(spk_out, targets):

    T, B, C = spk_out.shape
    device = spk_out.device

    #które neurony strzeliły:
    has_spike = spk_out.any(dim=0)
    #indeks pierwszego wystąpienia maksymalnej wartości, czyli 1
    first_spike_time = spk_out.argmax(dim = 0).float()
    #wszystkie neurony które nie strzeliły mają ogromną wartość T+100, przez co będą ignorowane
    first_spike_time[~has_spike] = float(T+100)
 
    _, idx = first_spike_time.min(dim = 1)

    B = targets.shape[0]
    batch_idx = torch.arange(B, device=targets.device)

    true_time = first_spike_time[batch_idx, targets] #czas pierwszego spike'a targeta

    wrong_time = first_spike_time.clone()
    wrong_time[batch_idx, targets] = T + 1000 #najwcześniejszy spike spośród złych klas

    best_wrong_time, _ = wrong_time.min(dim=1)
    
    target_active = has_spike[batch_idx, targets]

    has_any_spike = spk_out.any(dim=(0, 2))

    if target_active.any():           
        print(f"Target earlier than wrong: {(true_time[target_active] < best_wrong_time[target_active]).float().mean().item() * 100:.4f}%")
        print(f"Próbek bez żadnego spike'a: {has_any_spike.logical_not().sum().item()} / {B}")
        print(f"------------------------------------------------")

        
    accuracy = (idx == targets).float().mean().item()

    return accuracy

def measure_accuracy(model, dataloader):
  with torch.no_grad():
    model.eval()
    length = 0
    temporal_accuracy = 0

    for i, (events, targets) in enumerate(dataloader):

        #if i >= 5:
         #   break
        if events is None:
            continue

        events = events.to(device, dtype = torch.float, non_blocking=True)
        targets = targets.to(device, non_blocking=True).long()

        functional.reset_net(model)
        spk_rec = model(events)

        print(f"Procent aktywności sieci: {spk_rec.mean() * 100:.4f}%")
        
        spike_count = spk_rec.sum(dim=0)  # [B, C]
        B = targets.shape[0]

        target_count = spike_count[torch.arange(B, device=targets.device),targets]

        wrong_count = spike_count.clone()
        wrong_count[torch.arange(B, device=targets.device),targets] = 0
        best_wrong_count = wrong_count.max(dim=1).values

        print(f"------------------------------------------------")

        print(f"Target spike mean count: {target_count.float().mean().item():.4f}")

        print(f"Best wrong spike mean count: {best_wrong_count.float().mean().item():.4f}")

        temporal_accuracy += SF.accuracy_temporal(spk_rec, targets)
        length += 1
      
    accuracy = (temporal_accuracy * 100 / length) if length > 0 else 0.0
    print(f'Accuracy obliczone: {accuracy:.4f}%')

    return accuracy

# Profiler

In [19]:
import snntorch.functional as SF

In [20]:
convnet = ConvNet(2, 4, beta = 0.9, threshold = 0.5).to(device)
optimizer = torch.optim.Adam(convnet.parameters(), lr=1e-3, betas=(0.9, 0.999))
criterion = SF.ce_temporal_loss()

import torch.profiler as profiler
from snntorch import utils
def train_step(model, dataloader, optimizer, criterion, prof):
    model.train()
    
    for i, batch in enumerate(dataloader):
        if batch is None:
            continue

        inputs, labels = batch

        optimizer.zero_grad()
        functional.reset_net(model)

        inputs = inputs.to(device, dtype = torch.float, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()

        outputs  = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        if prof is not None:
            prof.step()
        if i >= 10:
            break

with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, 
    profiler.ProfilerActivity.CUDA],
    schedule=profiler.schedule(
        wait=1,
        warmup=1,
        active=9
    ),
    on_trace_ready=profiler.tensorboard_trace_handler("./log"),
    profile_memory=True,      
    with_stack=True 
) as prof:
    train_step(convnet, train_loader, optimizer, criterion, prof)

print(prof.key_averages().table(sort_by="cuda_time_total"))

# Optuna

In [21]:
from tqdm import tqdm
import optuna

In [22]:
n_trials = 20

In [23]:
import snntorch.functional as SF

In [34]:
def objective(trial):

    lr = trial.suggest_float('lr', 5e-5, 5e-4, log=True)
    beta = trial.suggest_float('beta', 0.97, 0.99)
    threshold = trial.suggest_float('threshold', 0.01, 0.03, log=True)

    model = ConvNet(in_ch=2, out_ch=16, beta=beta, threshold = threshold).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=1e-4)
    loss = SF.ce_temporal_loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',patience=5,factor=0.5)

    print(f"Trial {trial.number}")

    epochs_per_trial = 5

    for epoch in range(epochs_per_trial):
            model.train()
            step = 0

            for i,  (events, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", total=20)):
                if i >= 20:
                    break

                if events is None:
                    continue

                events = events.to(device, dtype = torch.float, non_blocking=True)
                targets = targets.to(device, non_blocking=True).long()

                optimizer.zero_grad(set_to_none=True)
                functional.reset_net(model)
                
                spk_rec = model(events)
                                
                spikes_percentage = spk_rec.mean() * 100
                print(f"Procent aktywności sieci: {spikes_percentage:.4f}%")

                loss_val = loss(spk_rec, targets) + 0.5 * (spk_rec.mean() - 0.1) ** 2
                loss_val.backward()
                                
                optimizer.step()

                step += 1

            print(f'Walidacja')
            val_acc = measure_accuracy(model, val_loader)
            
            scheduler.step(val_acc)
            trial.report(val_acc, step=epoch)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        
    return val_acc

In [37]:
sampler = optuna.samplers.TPESampler()
pruner = optuna.pruners.SuccessiveHalvingPruner(min_resource=1, reduction_factor=4)
study = optuna.create_study(study_name="14", sampler=sampler, direction='maximize', pruner=pruner, 
                            storage="sqlite:///14.db", load_if_exists=True)

study.optimize(objective, n_trials=n_trials)

df = study.trials_dataframe()

print(f"Best accuracy: {study.best_trial.value:.4f}")
print(f"Best config: {study.best_params}")

[I 2026-09-16 17:51:46,876] A new study created in RDB with name: 14


Trial 0


Epoch 1:   0%|          | 0/20 [00:00<?, ?it/s]

Procent aktywności sieci: 7.3040%


Epoch 1:   5%|▌         | 1/20 [00:01<00:23,  1.26s/it]

...

Procent aktywności sieci: 7.3293%


Epoch 2:  20%|██        | 4/20 [00:04<00:18,  1.18s/it]

Procent aktywności sieci: 6.9183%


Epoch 2:  25%|██▌       | 5/20 [00:06<00:18,  1.24s/it]
[W 2026-09-16 17:52:25,084] Trial 0 failed with parameters: {'lr': 7.683357490512026e-05, 'beta': 0.9792801801092614, 'threshold': 0.011544126162598776} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Veronika\AppData\Local\Temp\ipykernel_20068\3326930534.py", line 20, in objective
    for i,  (events, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", total=20)):
  File "c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\std.py", line 1186, in __iter__
    for obj in iterable:
  File "c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py", line 741, in __next__
    data = self._next_data()
  File "c:\Users\Vero

KeyboardInterrupt: 

In [26]:
study = optuna.load_study(
    study_name="5",
    storage="sqlite:///5.db"
)

In [27]:
params = study.best_params
print(f'Best accuracy: {study.best_trial.value:.2f}%, params: {params}')

Best accuracy: 20.31%, params: {'lr': 9.940843206716497e-05, 'beta': 0.983257626695972, 'threshold': 0.028083462568611824}


In [ ]:
study = optuna.load_study(
    study_name="24",
    storage="sqlite:///24.db"
)

In [ ]:
params = study.best_params
print(f'Best accuracy: {study.best_trial.value:.2f}%, params: {params}')

Best accuracy: 20.31%, params: {'lr': 7.551960796243714e-05, 'beta': 0.9862456903907272, 'threshold': 0.01597311854267052}


# Model

In [ ]:
model = ConvNet(in_ch=2, out_ch=8, beta = params['beta'], 
                threshold=params['threshold']).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr= params['lr'], betas=(0.9, 0.999))
loss = SF.ce_temporal_loss()

num_epochs = 20
loss_hist = []
acc_hist = []
best_accuracy = None

for epoch in range(num_epochs):
    model.train()
    step = 0
    epoch_loss = 0

    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        if batch is None:
            continue
        #if step >= 20:
        #    break
        events, targets = batch

        events = events.to(device, dtype = torch.float, non_blocking=True)
        targets = targets.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)
        functional.reset_net(model)
        
        spk_rec = model(events)        

        batch_loss = loss(spk_rec, targets) 

        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        step += 1
        epoch_loss += batch_loss

    train_loss = epoch_loss / step
    loss_hist.append(train_loss)

    model.eval()
    temporal_accuracy = 0

    with torch.no_grad():
        for events, targets in iter(val_loader):
            events = events.to(device, dtype = torch.float, non_blocking=True)
            targets = targets.to(device, non_blocking=True).long()

            functional.reset_net(model)

            spk_rec = model(events)

            T, B, C = spk_rec.shape
            device = spk_rec.device

            has_spike = spk_rec.any(dim = 0)

            first_spike_time = spk_rec.argmax(dim = 0).float()
            first_spike_time[~has_spike] = float(T+100)

            _, idx = first_spike_time.min(dim = 1)
            has_any_spike = spk_rec.any(dim=(0, 2))
            idx[~has_any_spike] = -1

            accuracy = (idx == targets).float().mean().item()

            temporal_accuracy += accuracy
                         
    accuracy = (temporal_accuracy * 100 / len(val_loader))
    acc_hist.append(accuracy)


    print(f"Epoch {epoch+1}/{num_epochs} \t Train Loss: {train_loss:.4f} \t Val Accuracy: {accuracy:.2f}%")


    if best_accuracy is None or accuracy > best_accuracy:
        best_accuracy = accuracy
        no_improvement_count = 0
    else:
        no_improvement_count += 1

        if no_improvement_count >= 5:
            print(f"Stopping early at epoch{epoch}")
            break

Epoch 1: 100%|██████████| 312/312 [17:57<00:00,  3.45s/it]


Epoch 1/20 	 Train Loss: 4.5694 	 Val Accuracy: 27.19%


Epoch 2: 100%|██████████| 312/312 [09:28<00:00,  1.82s/it]


Epoch 2/20 	 Train Loss: 4.0019 	 Val Accuracy: 28.34%


Epoch 3: 100%|██████████| 312/312 [07:12<00:00,  1.39s/it]


Epoch 3/20 	 Train Loss: 3.9892 	 Val Accuracy: 23.34%


Epoch 4:   1%|▏         | 4/312 [00:05<07:20,  1.43s/it]


KeyboardInterrupt: 